# Instacart Customer Segmentation and Split

This notebook builds simple customer segments and prepares a clean chronological split for rule mining evaluation.

## Goals
- Build customer features from prior orders
- Create customer segments (rare, frequent, heavy)
- Merge segments into basket-level data
- Create a chronological split by user:
  - train_rules (older prior orders)
  - validation (last prior order)
  - test_final (train order)

## Output
This notebook saves segmented order-level tables used in basket creation and rule mining.

In [16]:
import os
import pandas as pd
import numpy as np

In [17]:
def load_instacart_tables():
    """
    Load the main Instacart tables.
    """
    tables = {
        "orders": pd.read_csv("../data/orders.csv"),
        "order_products__prior": pd.read_csv("../data/order_products__prior.csv"),
        "order_products__train": pd.read_csv("../data/order_products__train.csv"),
        "products": pd.read_csv("../data/products.csv"),
        "aisles": pd.read_csv("../data/aisles.csv"),
        "departments": pd.read_csv("../data/departments.csv"),
    }
    return tables


def enrich_products(products, aisles, departments):
    """
    Merge products with aisle and department names.
    """
    df = products.merge(aisles, on="aisle_id", how="left")
    df = df.merge(departments, on="department_id", how="left")
    return df


def build_order_level_prior(orders, op_prior):
    """
    Build prior order-level table with basket size.
    """
    prior_orders = orders[orders["eval_set"] == "prior"].copy()

    basket_sizes = op_prior.groupby("order_id").size().reset_index(name="basket_size")
    basket_reorder = op_prior.groupby("order_id")["reordered"].mean().reset_index(name="reorder_rate_order")

    prior_order_level = prior_orders.merge(basket_sizes, on="order_id", how="left")
    prior_order_level = prior_order_level.merge(basket_reorder, on="order_id", how="left")

    prior_order_level["basket_size"] = prior_order_level["basket_size"].fillna(0)
    prior_order_level["reorder_rate_order"] = prior_order_level["reorder_rate_order"].fillna(0)

    return prior_order_level


def build_user_features(prior_order_level):
    """
    Build simple user features from prior orders.
    """
    user_features = prior_order_level.groupby("user_id").agg(
        n_prior_orders=("order_id", "count"),
        total_items_prior=("basket_size", "sum"),
        avg_basket_size_prior=("basket_size", "mean"),
        basket_size_std=("basket_size", "std"),
        avg_days_since_prior=("days_since_prior_order", "mean"),
        avg_reorder_rate=("reorder_rate_order", "mean"),
    ).reset_index()

    user_features["basket_size_std"] = user_features["basket_size_std"].fillna(0)
    user_features["avg_days_since_prior"] = user_features["avg_days_since_prior"].fillna(0)

    return user_features


def assign_segments(user_features):
    """
    Assign customer segments using total prior items.
    """
    df = user_features.copy()

    q1 = df["total_items_prior"].quantile(0.33)
    q2 = df["total_items_prior"].quantile(0.66)

    def segment_label(x):
        if x <= q1:
            return "rare"
        elif x <= q2:
            return "frequent"
        else:
            return "heavy"

    df["segment"] = df["total_items_prior"].apply(segment_label)
    return df


def segment_summary(user_features_seg):
    """
    Build a summary table by segment.
    """
    out = user_features_seg.groupby("segment").agg(
        n_users=("user_id", "count"),
        avg_n_prior_orders=("n_prior_orders", "mean"),
        avg_total_items_prior=("total_items_prior", "mean"),
        avg_basket_size_prior=("avg_basket_size_prior", "mean"),
        avg_reorder_rate=("avg_reorder_rate", "mean"),
    ).reset_index()

    order_map = {"rare": 0, "frequent": 1, "heavy": 2}
    out["segment_order"] = out["segment"].map(order_map)
    out = out.sort_values("segment_order").drop(columns="segment_order").reset_index(drop=True)
    return out


def build_prior_train_order_tables(orders, user_segments):
    """
    Build order tables with segment labels for prior and train.
    """
    order_cols = [
        "order_id",
        "user_id",
        "eval_set",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
    ]

    base_orders = orders[order_cols].copy()
    base_orders = base_orders.merge(
        user_segments[["user_id", "segment"]],
        on="user_id",
        how="left",
    )

    prior_orders = base_orders[base_orders["eval_set"] == "prior"].copy()
    train_orders = base_orders[base_orders["eval_set"] == "train"].copy()

    return prior_orders, train_orders


def build_chronological_split(prior_orders, train_orders):
    """
    Create train_rules, validation, and test_final split.
    """
    prior_sorted = prior_orders.sort_values(["user_id", "order_number"]).copy()

    prior_max = prior_sorted.groupby("user_id")["order_number"].max().reset_index(name="max_prior_order_number")
    prior_sorted = prior_sorted.merge(prior_max, on="user_id", how="left")

    prior_counts = prior_sorted.groupby("user_id")["order_id"].count().reset_index(name="n_prior_orders")
    prior_sorted = prior_sorted.merge(prior_counts, on="user_id", how="left")

    # Validation = last prior order
    validation_orders = prior_sorted[
        prior_sorted["order_number"] == prior_sorted["max_prior_order_number"]
    ].copy()

    # Train_rules = prior orders before the validation order
    train_rules_orders = prior_sorted[
        prior_sorted["order_number"] < prior_sorted["max_prior_order_number"]
    ].copy()

    # Keep users with at least 2 prior orders for validation logic
    valid_users = set(prior_counts[prior_counts["n_prior_orders"] >= 2]["user_id"].tolist())

    validation_orders = validation_orders[validation_orders["user_id"].isin(valid_users)].copy()
    train_rules_orders = train_rules_orders[train_rules_orders["user_id"].isin(valid_users)].copy()

    # Test_final = train order for users present in valid split
    test_final_orders = train_orders[train_orders["user_id"].isin(valid_users)].copy()

    return train_rules_orders, validation_orders, test_final_orders


def split_summary(train_rules_orders, validation_orders, test_final_orders):
    """
    Build a simple summary of split tables.
    """
    rows = []

    for name, df in [
        ("train_rules", train_rules_orders),
        ("validation", validation_orders),
        ("test_final", test_final_orders),
    ]:
        rows.append({
            "dataset": name,
            "n_orders": len(df),
            "n_users": df["user_id"].nunique(),
            "avg_order_number": df["order_number"].mean(),
            "segment_rare_share": (df["segment"] == "rare").mean(),
            "segment_frequent_share": (df["segment"] == "frequent").mean(),
            "segment_heavy_share": (df["segment"] == "heavy").mean(),
        })

    return pd.DataFrame(rows)


def segment_distribution_in_split(df, name):
    """
    Count orders by segment for one split table.
    """
    out = df["segment"].value_counts(dropna=False).reset_index()
    out.columns = ["segment", "n_orders"]
    out["share"] = out["n_orders"] / out["n_orders"].sum()
    out["dataset"] = name

    order_map = {"rare": 0, "frequent": 1, "heavy": 2}
    out["segment_order"] = out["segment"].map(order_map).fillna(999)
    out = out.sort_values("segment_order").drop(columns="segment_order").reset_index(drop=True)
    return out


def save_segmentation_outputs(user_features_seg, train_rules_orders, validation_orders, test_final_orders):
    """
    Save outputs for the next notebooks.
    """
    os.makedirs("../outputs", exist_ok=True)

    user_features_seg.to_csv("../outputs/customer_features_with_segments.csv", index=False)
    train_rules_orders.to_csv("../outputs/orders_train_rules.csv", index=False)
    validation_orders.to_csv("../outputs/orders_validation.csv", index=False)
    test_final_orders.to_csv("../outputs/orders_test_final.csv", index=False)

In [18]:
# Load data
tables = load_instacart_tables()

orders = tables["orders"]
op_prior = tables["order_products__prior"]
op_train = tables["order_products__train"]
products = tables["products"]
aisles = tables["aisles"]
departments = tables["departments"]

In [19]:
# Enrich products for simple category checks
products_enriched = enrich_products(products, aisles, departments)
display(products_enriched.head())

,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry


In [20]:
# Build prior order-level data
prior_order_level = build_order_level_prior(orders, op_prior)
display(prior_order_level.head())

# Build user features
user_features = build_user_features(prior_order_level)
display(user_features.head())

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,basket_size,reorder_rate_order
0,2539329,1,prior,1,2,8,NaN,5,0.000
1,2398795,1,prior,2,3,7,15.0,6,0.500
2,473747,1,prior,3,3,12,21.0,5,0.600
3,2254736,1,prior,4,4,7,29.0,5,1.000
4,431534,1,prior,5,4,15,28.0,8,0.625


,user_id,n_prior_orders,total_items_prior,avg_basket_size_prior,basket_size_std,avg_days_since_prior,avg_reorder_rate
0,1,10,59,5.900000,1.523884,19.555556,0.705833
1,2,14,195,13.928571,5.717238,15.230769,0.447961
2,3,12,88,7.333333,2.103388,12.090909,0.658817
3,4,5,18,3.600000,2.073644,13.750000,0.028571
4,5,4,37,9.250000,3.095696,13.333333,0.377778


In [21]:
# Assign segments
user_features_seg = assign_segments(user_features)
display(user_features_seg.head())

# Show segment summary
display(segment_summary(user_features_seg))

,user_id,n_prior_orders,total_items_prior,avg_basket_size_prior,basket_size_std,avg_days_since_prior,avg_reorder_rate,segment
0,1,10,59,5.900000,1.523884,19.555556,0.705833,frequent
1,2,14,195,13.928571,5.717238,15.230769,0.447961,heavy
2,3,12,88,7.333333,2.103388,12.090909,0.658817,frequent
3,4,5,18,3.600000,2.073644,13.750000,0.028571,rare
4,5,4,37,9.250000,3.095696,13.333333,0.377778,rare


,segment,n_users,avg_n_prior_orders,avg_total_items_prior,avg_basket_size_prior,avg_reorder_rate
0,rare,68270,5.471539,28.255368,6.079717,0.324440
1,frequent,68178,10.402857,86.063114,10.601077,0.425561
2,heavy,69761,30.562721,353.175614,13.105949,0.617899


In [22]:
# Merge segments into order-level tables
prior_orders_seg, train_orders_seg = build_prior_train_order_tables(orders, user_features_seg)

display(prior_orders_seg.head())
display(train_orders_seg.head())

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,segment
0,2539329,1,prior,1,2,8,NaN,frequent
1,2398795,1,prior,2,3,7,15.0,frequent
2,473747,1,prior,3,3,12,21.0,frequent
3,2254736,1,prior,4,4,7,29.0,frequent
4,431534,1,prior,5,4,15,28.0,frequent


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,segment
10,1187899,1,train,11,4,8,14.0,frequent
25,1492625,2,train,15,1,11,30.0,heavy
49,2196797,5,train,5,0,11,6.0,rare
74,525192,7,train,21,2,11,6.0,heavy
78,880375,8,train,4,1,14,10.0,rare


In [23]:
# Build chronological split
train_rules_orders, validation_orders, test_final_orders = build_chronological_split(
    prior_orders_seg,
    train_orders_seg,
)

# Show split summary
display(split_summary(train_rules_orders, validation_orders, test_final_orders))

,dataset,n_orders,n_users,avg_order_number,segment_rare_share,segment_frequent_share,segment_heavy_share
0,train_rules,3008665,206209,17.300775,0.101464,0.213074,0.685462
1,validation,206209,206209,15.590367,0.331072,0.330626,0.338302
2,test_final,131209,131209,16.603937,0.331212,0.330000,0.338788


In [24]:
# Show segment distribution in each split
display(segment_distribution_in_split(train_rules_orders, "train_rules"))
display(segment_distribution_in_split(validation_orders, "validation"))
display(segment_distribution_in_split(test_final_orders, "test_final"))

,segment,n_orders,share,dataset
0,rare,305272,0.101464,train_rules
1,frequent,641068,0.213074,train_rules
2,heavy,2062325,0.685462,train_rules


,segment,n_orders,share,dataset
0,rare,68270,0.331072,validation
1,frequent,68178,0.330626,validation
2,heavy,69761,0.338302,validation


,segment,n_orders,share,dataset
0,rare,43458,0.331212,test_final
1,frequent,43299,0.330000,test_final
2,heavy,44452,0.338788,test_final


In [25]:
# Simple checks for split consistency
users_train_rules = set(train_rules_orders["user_id"].unique())
users_validation = set(validation_orders["user_id"].unique())
users_test_final = set(test_final_orders["user_id"].unique())

split_checks = pd.DataFrame([{
    "users_in_train_rules": len(users_train_rules),
    "users_in_validation": len(users_validation),
    "users_in_test_final": len(users_test_final),
    "train_rules_validation_same_users": users_train_rules == users_validation,
    "validation_test_same_users": users_validation == users_test_final,
    "train_rules_test_same_users": users_train_rules == users_test_final,
}])
display(split_checks)

,users_in_train_rules,users_in_validation,users_in_test_final,train_rules_validation_same_users,validation_test_same_users,train_rules_test_same_users
0,206209,206209,131209,True,False,False


In [26]:
# Check chronology for a small sample of users
sample_users = validation_orders["user_id"].drop_duplicates().head(10).tolist()

chronology_check = []
for uid in sample_users:
    tr_max = train_rules_orders[train_rules_orders["user_id"] == uid]["order_number"].max()
    va_num = validation_orders[validation_orders["user_id"] == uid]["order_number"].min()
    te_num = test_final_orders[test_final_orders["user_id"] == uid]["order_number"].min()

    chronology_check.append({
        "user_id": uid,
        "max_train_rules_order_number": tr_max,
        "validation_order_number": va_num,
        "test_final_order_number": te_num,
        "is_ordered": (tr_max < va_num) and (va_num < te_num),
    })

chronology_check_df = pd.DataFrame(chronology_check)
display(chronology_check_df)

,user_id,max_train_rules_order_number,validation_order_number,test_final_order_number,is_ordered
0,1,9,10,11.0,True
1,2,13,14,15.0,True
2,3,11,12,NaN,False
3,4,4,5,NaN,False
4,5,3,4,5.0,True
5,6,2,3,NaN,False
6,7,19,20,21.0,True
7,8,2,3,4.0,True
8,9,2,3,4.0,True
9,10,4,5,6.0,True


In [27]:
# Save outputs for next notebooks
save_segmentation_outputs(
    user_features_seg,
    train_rules_orders,
    validation_orders,
    test_final_orders,
)

In [28]:
# Final summary
final_summary = pd.DataFrame([{
    "n_users_segmented": user_features_seg["user_id"].nunique(),
    "n_train_rules_orders": len(train_rules_orders),
    "n_validation_orders": len(validation_orders),
    "n_test_final_orders": len(test_final_orders),
    "outputs_saved": True,
}])
display(final_summary)

,n_users_segmented,n_train_rules_orders,n_validation_orders,n_test_final_orders,outputs_saved
0,206209,3008665,206209,131209,True
